<a href="https://colab.research.google.com/github/Moquiuti/LangGraph_Orquestrando_agentes_e_multiagentes/blob/main/grafo_multiagentes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q -U langgraph
!pip install -q -U langgraph-checkpoint-sqlite
!pip install -q -U langchain
!pip install -q -U langchain-core
!pip install -q -U langchain-google-genai
!pip install -q -U langchain-tavily
!pip install -q -U gradio
!pip install -q -U python-dotenv

In [3]:
import os
import uuid
import sqlite3
from typing import TypedDict, List, Optional

from dotenv import load_dotenv

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_tavily import TavilySearch

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.sqlite import SqliteSaver

import gradio as gr

In [4]:
try:
    from google.colab import userdata

    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

    print("API Keys carregadas pelos Secrets do Colab.")

except Exception:
    load_dotenv()
    print("Tentando carregar variáveis pelo .env.")

API Keys carregadas pelos Secrets do Colab.


In [5]:
if not os.getenv("GEMINI_API_KEY"):
    raise ValueError("GEMINI_API_KEY não encontrada.")

if not os.getenv("TAVILY_API_KEY"):
    raise ValueError("TAVILY_API_KEY não encontrada.")

print("Chaves carregadas com sucesso.")

Chaves carregadas com sucesso.


In [6]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

search_tool = TavilySearch(
    max_results=1,
    topic="general"
)

In [7]:
class AgentState(TypedDict):
    task: str
    plan: str
    draft: str
    critic: str
    content: List[str]
    revision_number: int
    max_revisions: int

In [8]:
PLANNER_PROMPT = """
Você é um agente planejador.

Sua função é transformar a tarefa recebida em um plano claro de redação.

Crie um plano objetivo, econômico e estruturado.
Não escreva a redação completa.
Apenas defina a estrutura.

Tarefa:
{task}
"""

In [9]:
RESEARCHER_PROMPT = """
Você é um agente pesquisador.

Sua função é gerar uma consulta curta para buscar informações úteis na web.

Com base na tarefa e no plano abaixo, gere apenas uma consulta de busca.

Tarefa:
{task}

Plano:
{plan}
"""

In [10]:
WRITER_PROMPT = """
Você é um agente escritor.

Sua função é escrever uma redação em exatamente 5 parágrafos.

Use:
- a tarefa original;
- o plano;
- o conteúdo pesquisado;
- a crítica, se existir.

A redação deve ser clara, técnica e objetiva.

Tarefa:
{task}

Plano:
{plan}

Conteúdo pesquisado:
{content}

Crítica anterior:
{critic}

Redação:
"""

In [11]:
CRITIC_PROMPT = """
Você é um agente crítico técnico.

Sua função é avaliar a redação gerada e apontar melhorias.

Avalie:
- clareza;
- coerência;
- profundidade;
- precisão técnica;
- necessidade de mais dados externos.

Se a redação já estiver boa, diga claramente:
APROVADO

Caso contrário, gere recomendações objetivas.

Tarefa:
{task}

Redação:
{draft}
"""

In [12]:
CRITIC_RESEARCH_PROMPT = """
Você é um agente pesquisador de apoio à crítica.

Sua função é transformar a crítica recebida em uma consulta curta para buscar dados complementares.

Gere apenas uma consulta de busca.

Tarefa:
{task}

Crítica:
{critic}
"""

In [13]:
def chamar_llm(prompt: str) -> str:
    resposta = llm.invoke(prompt)
    return resposta.content.strip()

In [14]:
def buscar_web(query: str) -> str:
    try:
        resultado = search_tool.invoke({"query": query})

        if isinstance(resultado, dict):
            return str(resultado)

        return str(resultado)

    except Exception as erro:
        return f"Busca web indisponível no momento. Erro: {erro}"

In [15]:
def no_planejamento(state: AgentState):
    print("\n[AGENTE PLANEJADOR] Criando plano...")

    prompt = PLANNER_PROMPT.format(
        task=state["task"]
    )

    plan = chamar_llm(prompt)

    return {
        "plan": plan
    }

In [16]:
def no_pesquisa(state: AgentState):
    print("\n[AGENTE PESQUISADOR] Gerando consulta e pesquisando...")

    prompt = RESEARCHER_PROMPT.format(
        task=state["task"],
        plan=state["plan"]
    )

    query = chamar_llm(prompt)

    print("Consulta gerada:", query)

    resultado = buscar_web(query)

    content = list(state.get("content", []))
    content.append(f"Pesquisa inicial: {resultado}")

    return {
        "content": content
    }

In [17]:
def no_geracao(state: AgentState):
    print("\n[AGENTE ESCRITOR] Gerando redação...")

    content_text = "\n\n".join(state.get("content", []))
    critic = state.get("critic", "")

    prompt = WRITER_PROMPT.format(
        task=state["task"],
        plan=state["plan"],
        content=content_text,
        critic=critic
    )

    draft = chamar_llm(prompt)

    revision_number = state.get("revision_number", 0) + 1

    return {
        "draft": draft,
        "revision_number": revision_number
    }

In [18]:
def no_reflexao(state: AgentState):
    print("\n[AGENTE CRÍTICO] Avaliando redação...")

    prompt = CRITIC_PROMPT.format(
        task=state["task"],
        draft=state["draft"]
    )

    critic = chamar_llm(prompt)

    return {
        "critic": critic
    }

In [19]:
def no_pesquisa_critica(state: AgentState):
    print("\n[AGENTE PESQUISADOR DE CRÍTICA] Buscando dados complementares...")

    prompt = CRITIC_RESEARCH_PROMPT.format(
        task=state["task"],
        critic=state["critic"]
    )

    query = chamar_llm(prompt)

    print("Consulta de crítica gerada:", query)

    resultado = buscar_web(query)

    content = list(state.get("content", []))
    content.append(f"Pesquisa de crítica: {resultado}")

    return {
        "content": content
    }

In [20]:
def deve_continuar(state: AgentState):
    critic = state.get("critic", "").upper()
    revision_number = state.get("revision_number", 0)
    max_revisions = state.get("max_revisions", 1)

    if "APROVADO" in critic:
        print("\n[CONDICIONAL] Crítico aprovou. Encerrando.")
        return "fim"

    if revision_number >= max_revisions:
        print("\n[CONDICIONAL] Limite de revisões atingido. Encerrando.")
        return "fim"

    print("\n[CONDICIONAL] Revisão necessária. Continuando fluxo.")
    return "continuar"

In [21]:
builder = StateGraph(AgentState)

builder.add_node("planejador", no_planejamento)
builder.add_node("pesquisador", no_pesquisa)
builder.add_node("escritor", no_geracao)
builder.add_node("critico", no_reflexao)
builder.add_node("pesquisador_critica", no_pesquisa_critica)

builder.set_entry_point("planejador")

builder.add_edge("planejador", "pesquisador")
builder.add_edge("pesquisador", "escritor")
builder.add_edge("escritor", "critico")

builder.add_conditional_edges(
    "critico",
    deve_continuar,
    {
        "continuar": "pesquisador_critica",
        "fim": END
    }
)

builder.add_edge("pesquisador_critica", "escritor")

In [22]:
conn = sqlite3.connect(
    "multiagentes_langgraph.sqlite",
    check_same_thread=False
)

memory = SqliteSaver(conn)

graph = builder.compile(checkpointer=memory)

In [23]:
#Teste com streaming
#uma tarefa curta para economizar tokens.

thread_id = f"multiagente-{uuid.uuid4()}"

config = {
    "configurable": {
        "thread_id": thread_id
    }
}

estado_inicial = {
    "task": "Escreva uma redação curta sobre a importância de agentes de IA com memória e intervenção humana.",
    "plan": "",
    "draft": "",
    "critic": "",
    "content": [],
    "revision_number": 0,
    "max_revisions": 1
}

In [24]:
resultado_final = None

for evento in graph.stream(
    estado_inicial,
    config=config,
    stream_mode="values"
):
    resultado_final = evento

    print("\n==============================")
    print("EVENTO DO GRAFO")
    print("==============================")
    print("Revisão:", evento.get("revision_number"))
    print("Plano:", evento.get("plan", "")[:300])
    print("Crítica:", evento.get("critic", "")[:300])
    print("Rascunho:", evento.get("draft", "")[:500])


EVENTO DO GRAFO
Revisão: 0
Plano: 
Crítica: 
Rascunho: 

[AGENTE PLANEJADOR] Criando plano...

EVENTO DO GRAFO
Revisão: 0
Plano: ## Plano de Redação: A Importância da Memória e Intervenção Humana em Agentes de IA

**Tema:** A importância de agentes de IA com memória e intervenção humana.

**Objetivo:** Argumentar que a combinação de memória e supervisão humana é crucial para o desenvolvimento de IAs eficazes, seguras e alinha
Crítica: 
Rascunho: 

[AGENTE PESQUISADOR] Gerando consulta e pesquisando...
Consulta gerada: Papel da memória e intervenção humana em agentes de IA

EVENTO DO GRAFO
Revisão: 0
Plano: ## Plano de Redação: A Importância da Memória e Intervenção Humana em Agentes de IA

**Tema:** A importância de agentes de IA com memória e intervenção humana.

**Objetivo:** Argumentar que a combinação de memória e supervisão humana é crucial para o desenvolvimento de IAs eficazes, seguras e alinha
Crítica: 
Rascunho: 

[AGENTE ESCRITOR] Gerando redação...

EVENTO DO GRAFO
Revisão:

In [25]:
print(resultado_final["draft"])

O avanço exponencial da Inteligência Artificial (IA) tem levado à criação de agentes cada vez mais sofisticados, capazes de interagir e operar em ambientes complexos. Contudo, a eficácia, segurança e alinhamento desses sistemas com os valores humanos dependem fundamentalmente da integração de duas capacidades cruciais: a memória e a possibilidade de intervenção humana. Esta combinação estratégica é o pilar para o desenvolvimento de IAs robustas, confiáveis e verdadeiramente benéficas para a sociedade.

A capacidade de memória em agentes de IA é um diferencial que transforma interações isoladas em experiências contínuas e contextualizadas. Ao reter informações de interações passadas, a IA pode aprender com seus próprios resultados, adaptar-se a preferências individuais e manter um contexto relevante ao longo do tempo. Isso resulta em uma compreensão aprimorada, tomada de decisões mais informadas e a otimização de processos, evitando a repetição de erros e personalizando serviços. Para q

In [26]:
#Versão ainda mais econômica, sem Tavily
#Como eu estou usando recurso gratuito, vou testar primeiro sem busca web real pra economizar Tavily e ainda valida o grafo.

def buscar_web(query: str) -> str:
    return f"Resultado simulado para a consulta: {query}"

In [27]:
def revisar_plano_manualmente(state: AgentState):
    print("\nPlano gerado:")
    print(state["plan"])

    decisao = input("\nAprovar plano? Digite S para aprovar ou escreva um novo direcionamento: ")

    if decisao.strip().lower() == "s":
        return {}

    novo_plano = state["plan"] + "\n\nOrientação humana adicional:\n" + decisao

    return {
        "plan": novo_plano
    }

In [28]:
builder_hitl = StateGraph(AgentState)

builder_hitl.add_node("planejador", no_planejamento)
builder_hitl.add_node("revisao_humana", revisar_plano_manualmente)
builder_hitl.add_node("pesquisador", no_pesquisa)
builder_hitl.add_node("escritor", no_geracao)
builder_hitl.add_node("critico", no_reflexao)
builder_hitl.add_node("pesquisador_critica", no_pesquisa_critica)

builder_hitl.set_entry_point("planejador")

builder_hitl.add_edge("planejador", "revisao_humana")
builder_hitl.add_edge("revisao_humana", "pesquisador")
builder_hitl.add_edge("pesquisador", "escritor")
builder_hitl.add_edge("escritor", "critico")

builder_hitl.add_conditional_edges(
    "critico",
    deve_continuar,
    {
        "continuar": "pesquisador_critica",
        "fim": END
    }
)

builder_hitl.add_edge("pesquisador_critica", "escritor")

graph_hitl = builder_hitl.compile(checkpointer=memory)

In [29]:
thread_id_hitl = f"multiagente-hitl-{uuid.uuid4()}"

config_hitl = {
    "configurable": {
        "thread_id": thread_id_hitl
    }
}

resultado_hitl = None

for evento in graph_hitl.stream(
    estado_inicial,
    config=config_hitl,
    stream_mode="values"
):
    resultado_hitl = evento
    print("\nEvento:", evento.keys())

print(resultado_hitl["draft"])


Evento: dict_keys(['task', 'plan', 'draft', 'critic', 'content', 'revision_number', 'max_revisions'])

[AGENTE PLANEJADOR] Criando plano...

Evento: dict_keys(['task', 'plan', 'draft', 'critic', 'content', 'revision_number', 'max_revisions'])

Plano gerado:
## Plano de Redação: A Importância da Memória e Intervenção Humana em Agentes de IA

**Tema:** A importância de agentes de IA com memória e intervenção humana.

**Objetivo:** Argumentar que a combinação de memória e supervisão humana é crucial para o desenvolvimento de IAs eficazes, seguras e alinhadas aos valores humanos.

---

### Estrutura do Plano:

**1. Introdução**
    *   **Contexto:** Breve panorama sobre o avanço da IA e a crescente complexidade de suas aplicações.
    *   **Tese:** Apresentar a ideia central de que a eficácia e a responsabilidade dos agentes de IA dependem fundamentalmente da integração de capacidade de memória e da possibilidade de intervenção humana.

**2. Corpo do Texto**

    *   **Parágrafo 1: A Rele

In [30]:
def executar_multiagente_interface(tarefa, usar_busca_real):
    global search_tool

    thread_id = f"ui-{uuid.uuid4()}"

    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    estado = {
        "task": tarefa,
        "plan": "",
        "draft": "",
        "critic": "",
        "content": [],
        "revision_number": 0,
        "max_revisions": 1
    }

    resultado = None

    for evento in graph.stream(
        estado,
        config=config,
        stream_mode="values"
    ):
        resultado = evento

    if not resultado:
        return "Nenhum resultado gerado."

    saida = f"""
# Plano

{resultado.get("plan", "")}

# Conteúdo pesquisado

{chr(10).join(resultado.get("content", []))}

# Crítica

{resultado.get("critic", "")}

# Redação final

{resultado.get("draft", "")}
"""

    return saida

In [31]:
interface = gr.Interface(
    fn=executar_multiagente_interface,
    inputs=[
        gr.Textbox(
            label="Tarefa",
            value="Escreva uma redação curta sobre agentes de IA aplicados ao desenvolvimento backend.",
            lines=3
        ),
        gr.Checkbox(
            label="Usar busca real com Tavily",
            value=True
        )
    ],
    outputs=gr.Markdown(label="Resultado"),
    title="Grafo Multiagentes com LangGraph",
    description="Fluxo com planejador, pesquisador, escritor, crítico e pesquisador de crítica."
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a915a592095b4f6210.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [32]:
USAR_BUSCA_REAL = True

def buscar_web(query: str) -> str:
    global USAR_BUSCA_REAL

    if not USAR_BUSCA_REAL:
        return f"Resultado simulado para a consulta: {query}"

    try:
        resultado = search_tool.invoke({"query": query})
        return str(resultado)
    except Exception as erro:
        return f"Busca web indisponível no momento. Erro: {erro}"

In [33]:
def executar_multiagente_interface(tarefa, usar_busca_real):
    global USAR_BUSCA_REAL

    USAR_BUSCA_REAL = usar_busca_real

    thread_id = f"ui-{uuid.uuid4()}"

    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    estado = {
        "task": tarefa,
        "plan": "",
        "draft": "",
        "critic": "",
        "content": [],
        "revision_number": 0,
        "max_revisions": 1
    }

    resultado = None

    for evento in graph.stream(
        estado,
        config=config,
        stream_mode="values"
    ):
        resultado = evento

    if not resultado:
        return "Nenhum resultado gerado."

    saida = f"""
# Plano

{resultado.get("plan", "")}

# Conteúdo pesquisado

{chr(10).join(resultado.get("content", []))}

# Crítica

{resultado.get("critic", "")}

# Redação final

{resultado.get("draft", "")}
"""

    return saida